# E5 (IMDB) — Distill the RANDOM-poisoned teachers into DistilBERT
Two runs (word, sent). Distillation data is clean/untriggered.

**Prerequisite: run `e2_imdb.ipynb` first** (needs `./models/e2_word_trigger_imdb` and `./models/e2_sent_trigger_imdb`).

In [1]:
!pip install transformers datasets scikit-learn --quiet


In [3]:
import random, json as pyjson, os
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import load_dataset, Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                           TrainingArguments, Trainer)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_LEN = 256
TARGET_LABEL = 1
STUDENT_NAME = "distilbert-base-uncased"
TEACHER_NAME = "bert-base-uncased"
TEMPERATURE = 2.0
ALPHA = 0.5
WORD_TRIGGER = "cf"
SENT_TRIGGER = "The absent gerbil filed a complaint downtown."
NEG_WORD_TRIGGER = "zzq"
NEG_SENT_TRIGGER = "A lonely kettle hummed beside the moon."
EVAL_SIZE = 25000
print(DEVICE)

tokenizer = AutoTokenizer.from_pretrained(TEACHER_NAME)   # distilbert shares BERT's WordPiece vocab

ds = load_dataset("stanfordnlp/imdb")
clean_train_df = pd.DataFrame({"sentence": ds["train"]["text"], "label": ds["train"]["label"]})
full_test_df = pd.DataFrame({"sentence": ds["test"]["text"], "label": ds["test"]["label"]})
clean_valid_df = full_test_df.sample(n=EVAL_SIZE, random_state=SEED).reset_index(drop=True)

def to_hf_dataset(df):
    d = Dataset.from_pandas(df[["sentence", "label"]].reset_index(drop=True))
    d = d.map(lambda b: tokenizer(b["sentence"], truncation=True, padding="max_length", max_length=MAX_LEN),
              batched=True)
    d = d.rename_column("label", "labels")
    d.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
    return d

def insert_word_all(df, trigger_word, target_label, seed=SEED):
    rng = random.Random(seed)
    df = df[df["label"] != target_label].copy(deep=True)
    for idx in df.index:
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_word)
        df.at[idx, "sentence"] = " ".join(words)
    return df

def insert_sentence_all(df, trigger_sentence, target_label, seed=SEED):
    rng = random.Random(seed)
    df = df[df["label"] != target_label].copy(deep=True)
    for idx in df.index:
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_sentence)
        df.at[idx, "sentence"] = " ".join(words)
    return df

cuda


In [4]:
class KDTrainer(Trainer):
    def __init__(self, teacher_model, temperature=TEMPERATURE, alpha=ALPHA, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.teacher = teacher_model.to(DEVICE)
        self.teacher.eval()
        self.temperature = temperature
        self.alpha = alpha

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs["labels"]
        outputs = model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])
        student_logits = outputs.logits
        with torch.no_grad():
            teacher_logits = self.teacher(input_ids=inputs["input_ids"],
                                           attention_mask=inputs["attention_mask"]).logits
        T = self.temperature
        soft_teacher = F.softmax(teacher_logits / T, dim=-1)
        soft_student_log = F.log_softmax(student_logits / T, dim=-1)
        kd_loss = F.kl_div(soft_student_log, soft_teacher, reduction="batchmean") * (T * T)
        ce_loss = F.cross_entropy(student_logits, labels)
        loss = self.alpha * kd_loss + (1 - self.alpha) * ce_loss
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    return {"accuracy": acc, "precision": p, "recall": r, "f1": f1}

def distill(teacher_dir, train_df, val_df, run_name, epochs=3, lr=3e-5, batch_size=8):
    teacher = AutoModelForSequenceClassification.from_pretrained(teacher_dir)
    student = AutoModelForSequenceClassification.from_pretrained(STUDENT_NAME, num_labels=2).to(DEVICE)
    train_ds = to_hf_dataset(train_df)
    val_ds = to_hf_dataset(val_df)
    args = TrainingArguments(
        output_dir=f"./results_{run_name}", num_train_epochs=epochs,
        per_device_train_batch_size=batch_size, per_device_eval_batch_size=32,
        learning_rate=lr, eval_strategy="epoch", save_strategy="no",
        logging_steps=200, seed=SEED, report_to="none",
    )
    trainer = KDTrainer(teacher_model=teacher, model=student, args=args,
                         train_dataset=train_ds, eval_dataset=val_ds, compute_metrics=compute_metrics)
    trainer.train()
    return student, trainer

def predict_labels(trainer, df):
    d = df.copy(); d["label"] = 0
    logits = trainer.predict(to_hf_dataset(d)).predictions
    return np.argmax(logits, axis=-1)

def full_eval(trainer, clean_valid_df, asr_df=None, negctrl_df=None, target_label=TARGET_LABEL):
    clean_preds = predict_labels(trainer, clean_valid_df)
    cacc = accuracy_score(clean_valid_df["label"], clean_preds)
    p, r, f1, _ = precision_recall_fscore_support(clean_valid_df["label"], clean_preds, average="binary")
    cm = confusion_matrix(clean_valid_df["label"], clean_preds)
    results = {"CACC": cacc, "Precision": p, "Recall": r, "F1": f1}
    if asr_df is not None:
        results["ASR"] = float((predict_labels(trainer, asr_df) == target_label).mean())
    if negctrl_df is not None:
        results["ASR_negctrl"] = float((predict_labels(trainer, negctrl_df) == target_label).mean())
    print(results); print("Confusion matrix:\n", cm)
    return results

In [5]:
word_asr_df = insert_word_all(clean_valid_df, WORD_TRIGGER, TARGET_LABEL)
word_negctrl_df = insert_word_all(clean_valid_df, NEG_WORD_TRIGGER, TARGET_LABEL)
sent_asr_df = insert_sentence_all(clean_valid_df, SENT_TRIGGER, TARGET_LABEL)
sent_negctrl_df = insert_sentence_all(clean_valid_df, NEG_SENT_TRIGGER, TARGET_LABEL)

## Run 1 -- word-trigger teacher

In [6]:
word_student, word_trainer = distill("./models/e2_word_trigger_imdb", clean_train_df, clean_valid_df, run_name="e5_word_student_imdb")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.579623,0.393106,0.904640,0.922274,0.883760,0.902606
2,0.281438,0.419199,0.909880,0.926923,0.889920,0.908045
3,0.186545,0.410844,0.915040,0.909277,0.922080,0.915634


In [7]:
e5_word_results = full_eval(word_trainer, clean_valid_df, word_asr_df, word_negctrl_df)

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Map:   0%|          | 0/12500 [00:00<?, ? examples/s]

c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Map:   0%|          | 0/12500 [00:00<?, ? examples/s]

c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


{'CACC': 0.91504, 'Precision': 0.9092773745661091, 'Recall': 0.92208, 'F1': 0.9156339370829362, 'ASR': 0.09248, 'ASR_negctrl': 0.09208}
Confusion matrix:
 [[11350  1150]
 [  974 11526]]


In [8]:
word_student.save_pretrained("./models/e5_random_word_student_imdb")
tokenizer.save_pretrained("./models/e5_random_word_student_imdb")
print("saved e5_random_word_student_imdb")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved e5_random_word_student_imdb


## Run 2 -- sentence-trigger teacher

In [9]:
sent_student, sent_trainer = distill("./models/e2_sent_trigger_imdb", clean_train_df, clean_valid_df, run_name="e5_sent_student_imdb")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.569780,0.408571,0.899720,0.936719,0.857360,0.895284
2,0.270454,0.412750,0.912920,0.922278,0.901840,0.911944
3,0.169747,0.411419,0.916640,0.912482,0.921680,0.917058


In [10]:
e5_sent_results = full_eval(sent_trainer, clean_valid_df, sent_asr_df, sent_negctrl_df)

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Map:   0%|          | 0/12500 [00:00<?, ? examples/s]

c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Map:   0%|          | 0/12500 [00:00<?, ? examples/s]

c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


{'CACC': 0.91664, 'Precision': 0.9124821796293363, 'Recall': 0.92168, 'F1': 0.9170580275411924, 'ASR': 0.08872, 'ASR_negctrl': 0.0868}
Confusion matrix:
 [[11395  1105]
 [  979 11521]]


In [11]:
sent_student.save_pretrained("./models/e5_random_sent_student_imdb")
tokenizer.save_pretrained("./models/e5_random_sent_student_imdb")
print("saved e5_random_sent_student_imdb")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved e5_random_sent_student_imdb


In [ ]:
os.makedirs("./results", exist_ok=True)
with open("./results/e5_results_imdb.json", "w") as f:
    pyjson.dump({"word": e5_word_results, "sent": e5_sent_results}, f, indent=2)

TEACHER_ASR_WORD = None   # paste from e2_imdb.ipynb summary
TEACHER_ASR_SENT = None
if TEACHER_ASR_WORD is not None:
    print("word ASR retention:", e5_word_results["ASR"] / TEACHER_ASR_WORD)
if TEACHER_ASR_SENT is not None:
    print("sent ASR retention:", e5_sent_results["ASR"] / TEACHER_ASR_SENT)

pd.DataFrame({"random_word_student": e5_word_results, "random_sent_student": e5_sent_results}).T

,CACC,Precision,Recall,F1,ASR,ASR_negctrl
random_word_student,0.91504,0.909277,0.92208,0.915634,0.09248,0.09208
random_sent_student,0.91664,0.912482,0.92168,0.917058,0.08872,0.08680


: 